# Underwater Waste Detection — Faster R-CNN Training

Two-stage detector: ResNet-50 FPN backbone + Region Proposal Network + ROI heads.

Why include Faster R-CNN?
- Classic anchor-based detector — good academic comparison point
- Handles large objects better than single-stage detectors
- Slower but often more precise bounding boxes
- torchvision implementation with pretrained ImageNet backbone

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────
!nvidia-smi
!pip install -q pycocotools

In [ ]:
# ── Cell 2: Mount Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys

DRIVE_BASE       = Path('/content/drive/MyDrive/underwater_waste_detection')
YOLO_DATASET_DIR = DRIVE_BASE / 'trashcan_yolo'
WEIGHTS_DIR      = DRIVE_BASE / 'weights'
WEIGHTS_DIR.mkdir(exist_ok=True)

REPO_DIR = Path('/content/underwater-waste-detection')
if not REPO_DIR.exists():
    !git clone https://github.com/omprxkash/underwater-waste-detection.git {REPO_DIR}
sys.path.insert(0, str(REPO_DIR / 'src'))
os.chdir(REPO_DIR)

In [ ]:
# ── Cell 3: Convert YOLO labels → Faster R-CNN pickle format ──────────────
# Faster R-CNN uses a pandas DataFrame loaded from pickle
# We'll build this from the YOLO-format labels we already have

import json
import pandas as pd
import numpy as np
from PIL import Image

CLASS_NAMES = ['background', 'trash', 'bio', 'rov']  # 0=bg, 1=trash, 2=bio, 3=rov

def yolo_to_rcnn_dataframe(images_dir, labels_dir):
    rows = []
    img_id = 0
    images_path = Path(images_dir)
    labels_path = Path(labels_dir)

    for img_file in sorted(images_path.glob('*.jpg')):
        lbl_file = labels_path / (img_file.stem + '.txt')
        if not lbl_file.exists():
            continue

        with Image.open(img_file) as im:
            img_w, img_h = im.size

        with open(lbl_file) as f:
            lines = f.read().strip().split('\n')

        for line in lines:
            if not line.strip():
                continue
            parts = line.split()
            cls_id = int(parts[0]) + 1  # shift: 0→1, 1→2, 2→3 (0 reserved for background)
            cx, cy, w_n, h_n = map(float, parts[1:])
            # convert YOLO normalized → COCO [x, y, w, h]
            w_px = w_n * img_w
            h_px = h_n * img_h
            x_px = (cx - w_n / 2) * img_w
            y_px = (cy - h_n / 2) * img_h
            rows.append({
                'image_id': img_id,
                'file_name': img_file.name,
                'bbox': [x_px, y_px, w_px, h_px],
                'category_id': cls_id,
                'img_width': img_w,
                'img_height': img_h,
            })
        img_id += 1

    return pd.DataFrame(rows)

print('Building training DataFrame...')
train_df = yolo_to_rcnn_dataframe(
    YOLO_DATASET_DIR / 'images' / 'train',
    YOLO_DATASET_DIR / 'labels' / 'train',
)
val_df = yolo_to_rcnn_dataframe(
    YOLO_DATASET_DIR / 'images' / 'val',
    YOLO_DATASET_DIR / 'labels' / 'val',
)

# Save as pickle for reuse
train_df.to_pickle(str(DRIVE_BASE / 'rcnn_train.pkl'))
val_df.to_pickle(str(DRIVE_BASE / 'rcnn_val.pkl'))

print(f'Train: {train_df["image_id"].nunique()} images, {len(train_df)} annotations')
print(f'Val:   {val_df["image_id"].nunique()} images, {len(val_df)} annotations')

In [ ]:
# ── Cell 4: Dataset class ─────────────────────────────────────────────────
import copy
import torch
from PIL import Image
import torchvision.transforms as T

class WasteDetectionDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, images_dir, transforms=None):
        self.df = dataframe
        self.images_dir = Path(images_dir)
        self.transforms = transforms
        self.image_ids = sorted(dataframe['image_id'].unique())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        rows = self.df[self.df['image_id'] == img_id]

        img_path = self.images_dir / rows['file_name'].iloc[0]
        image = Image.open(img_path).convert('RGB')

        boxes, labels = [], []
        for _, row in rows.iterrows():
            x, y, w, h = row['bbox']
            x2, y2 = x + w, y + h
            if x2 > x and y2 > y:
                boxes.append([x, y, x2, y2])
                labels.append(row['category_id'])

        boxes_t  = torch.as_tensor(boxes,  dtype=torch.float32) if boxes else torch.zeros((0, 4))
        labels_t = torch.as_tensor(labels, dtype=torch.int64)[:len(boxes)]
        area = (boxes_t[:, 3] - boxes_t[:, 1]) * (boxes_t[:, 2] - boxes_t[:, 0])

        target = {
            'boxes':    boxes_t,
            'labels':   labels_t,
            'image_id': torch.tensor([img_id]),
            'area':     area,
            'iscrowd':  torch.zeros(len(boxes), dtype=torch.int64),
        }

        if self.transforms:
            image = self.transforms(image)
        return image, target

def collate_fn(batch):
    return tuple(zip(*batch))

transform = T.Compose([T.ToTensor()])

train_set = WasteDetectionDataset(train_df, YOLO_DATASET_DIR / 'images' / 'train', transform)
val_set   = WasteDetectionDataset(val_df,   YOLO_DATASET_DIR / 'images' / 'val',   transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=4, shuffle=True,  num_workers=2, collate_fn=collate_fn)
val_loader   = torch.utils.data.DataLoader(val_set,   batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# ── Cell 5: Build model ───────────────────────────────────────────────────
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

NUM_CLASSES = 4  # background + trash + bio + rov

def build_detector(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT')
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f'Device: {device}')

rcnn_model = build_detector(NUM_CLASSES)
rcnn_model.to(device)

params = [p for p in rcnn_model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.002, momentum=0.9, weight_decay=0.0001)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

In [ ]:
# ── Cell 6: Training loop ─────────────────────────────────────────────────
import time

NUM_EPOCHS = 15
SAVE_PATH = str(WEIGHTS_DIR / 'faster_rcnn_best.pth')
epoch_losses = []
best_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    rcnn_model.train()
    running_loss = 0.0
    t_start = time.time()

    for batch_idx, (images, targets) in enumerate(train_loader):
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = rcnn_model(images, targets)
        total_loss = sum(loss_dict.values())

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        running_loss += total_loss.item()

        if batch_idx % 20 == 0:
            print(f'  Epoch {epoch+1}/{NUM_EPOCHS} | Batch {batch_idx}/{len(train_loader)} | Loss: {total_loss.item():.4f}')

    lr_scheduler.step()
    avg_loss = running_loss / len(train_loader)
    epoch_losses.append(avg_loss)
    elapsed = time.time() - t_start
    print(f'Epoch {epoch+1} done — avg loss: {avg_loss:.4f} ({elapsed:.0f}s)')

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(rcnn_model.state_dict(), SAVE_PATH)
        print(f'  -> Saved best model (loss={best_loss:.4f})')

print(f'\nTraining complete. Best model saved to: {SAVE_PATH}')

In [ ]:
# ── Cell 7: Plot loss curve ───────────────────────────────────────────────
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(range(1, len(epoch_losses)+1), epoch_losses, marker='o', linewidth=2, color='#2980b9')
plt.title('Faster R-CNN Training Loss', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(DRIVE_BASE / 'faster_rcnn_loss.png'), dpi=150)
plt.show()

In [ ]:
# ── Cell 8: Visualize predictions ─────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import random

RCNN_CLASS_COLORS = ['gray', '#e74c3c', '#27ae60', '#3498db']
RCNN_CLASS_NAMES  = ['bg', 'trash', 'bio', 'rov']
CONF_THRESHOLD    = 0.5

# Load best model
loaded_model = build_detector(NUM_CLASSES)
loaded_model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
loaded_model.to(device).eval()

val_images = list((YOLO_DATASET_DIR / 'images' / 'val').glob('*.jpg'))
sample_paths = random.sample(val_images, min(4, len(val_images)))

fig, axes = plt.subplots(1, len(sample_paths), figsize=(5*len(sample_paths), 5))
if len(sample_paths) == 1:
    axes = [axes]

for ax, img_path in zip(axes, sample_paths):
    pil_img = Image.open(img_path).convert('RGB')
    import torchvision.transforms as T
    tensor = T.ToTensor()(pil_img).unsqueeze(0).to(device)

    with torch.no_grad():
        preds = loaded_model(tensor)[0]

    ax.imshow(pil_img)
    for box, score, label in zip(preds['boxes'], preds['scores'], preds['labels']):
        if score.item() < CONF_THRESHOLD:
            continue
        x1, y1, x2, y2 = box.cpu().numpy()
        cls_idx = label.item()
        color = RCNN_CLASS_COLORS[cls_idx] if cls_idx < len(RCNN_CLASS_COLORS) else 'white'
        rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        name = RCNN_CLASS_NAMES[cls_idx] if cls_idx < len(RCNN_CLASS_NAMES) else '?'
        ax.text(x1, y1-4, f'{name} {score:.0%}', color=color, fontsize=8, fontweight='bold')

    ax.axis('off')
    ax.set_title(img_path.name[:20], fontsize=9)

plt.suptitle('Faster R-CNN Predictions (conf > 0.5)', fontweight='bold')
plt.tight_layout()
plt.savefig(str(DRIVE_BASE / 'faster_rcnn_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()